In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

In [ ]:
data = pd.read_csv("ADSI_Table_1A.2.csv")
data  

In [ ]:
data.columns
data.shape
# data.info
# data.describe 

# EDA

In [ ]:
rename_map = {
        "Sl. No."                              : "Sl_No",
        "State/UT/City"                        : "State_UT",
        "Road Accidents - Cases"               : "Road_Cases",
        "Road Accidents - Injured"             : "Road_Injured",
        "Road Accidents - Died"                : "Road_Died",
        "Railway Accidents - Cases"            : "Rail_Cases",
        "Railway Accidents - Injured"          : "Rail_Injured",
        "Railway Accidents - Died"             : "Rail_Died",
        "Railway Crossing Accidents - Cases"   : "Crossing_Cases",
        "Railway Crossing Accidents - Injured" : "Crossing_Injured",
        "Railway Crossing Accidents - Died"    : "Crossing_Died",
        "Total Traffic Accidents - Cases"      : "Total_Cases",
        "Total Traffic Accidents - Injured"    : "Total_Injured",
        "Total Traffic Accidents - Died"       : "Total_Died",
}
data.rename(columns = rename_map , inplace=True)
# data

In [ ]:
# Drop serial number if present
if "Sl_No" in data.columns:
    data.drop(columns=["Sl_No"], inplace=True)
# data 

In [ ]:
# Data types
print("\nColumn dtypes:")
data.dtypes

In [ ]:
# Missing values
data.isnull().sum()
# no missing values found. 

# Duplicate rows
dupes = data.duplicated().sum()
print(f"\n🔁  Duplicate rows : {dupes}") 

# Negative values (logically impossible for accident counts)
numeric_cols = data.select_dtypes(include=np.number).columns
neg_mask = (data[numeric_cols] < 0).any()
neg_mask[neg_mask].index.tolist() 

 # Basic stats
print("\n📊  Descriptive statistics:")
print(data[numeric_cols].describe().round(2).to_string())

# KEY INSIGHT SUMMARY

In [ ]:
def print_key_insights(data: pd.DataFrame) -> None:
    print("  KEY INSIGHTS")
 
    total_deaths  = data["Total_Died"].sum()
    total_cases   = data["Total_Cases"].sum() 
    overall_rate  = total_deaths / total_cases * 100 if total_cases else 0
 
    road_deaths   = data["Road_Died"].sum()
    rail_deaths   = data["Rail_Died"].sum()
    cross_deaths  = data["Crossing_Died"].sum()
 
    top_state     = data.loc[data["Total_Died"].idxmax(), "State_UT"]
    top_state_val = data["Total_Died"].max()
 
    df2 = data.copy()
    df2["Road_FR"] = df2["Road_Died"] / df2["Road_Cases"].replace(0, np.nan) * 100
    highest_fr    = df2.loc[df2["Road_FR"].idxmax(), "State_UT"]
    highest_fr_val= df2["Road_FR"].max()
 
    print(f"\n1. Total deaths        : {total_deaths:,}")
    print(f"2. Total cases         : {total_cases:,}")
    print(f"3. Overall fatality    : {overall_rate:.2f}%")
    print(f"4. Road deaths share   : {road_deaths/total_deaths*100:.1f}%")
    print(f"5. Railway share       : {rail_deaths/total_deaths*100:.1f}%")
    print(f"6. Crossing share      : {cross_deaths/total_deaths*100:.1f}%")
    print(f"7. Most deaths         : {top_state} ({top_state_val:,})")
    print(f"8. Highest fatality %  : {highest_fr} ({highest_fr_val:.1f}%)")
 
print_key_insights(data)

In [ ]:
def detect_outliers_iqr(data: pd.DataFrame, cols: list) -> pd.DataFrame:
    """Return a DataFrame summarising outlier counts per column."""
    results = []
    for col in cols:
        q1  = data[col].quantile(0.25)
        q3  = data[col].quantile(0.75)
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        outliers = data[(data[col] < lower) | (data[col] > upper)]
        results.append({ 
            "Column"   : col,
            "Q1"       : round(q1, 1),
            "Q3"       : round(q3, 1),
            "IQR"      : round(iqr, 1),
            "Lower fence": round(lower, 1),
            "Upper fence": round(upper, 1),
            "# Outliers": len(outliers),
            "Outlier states": ", ".join(outliers["State_UT"].tolist()[:3])
                              + ("…" if len(outliers) > 3 else ""),
        }) 
    return pd.DataFrame(results)
detect_outliers_iqr(data, numeric_cols)    

In [ ]:
def plot_correlation_heatmap(data: pd.DataFrame) -> None:
    numeric_df = data.select_dtypes(include=np.number)
    corr = numeric_df.corr()
 
    mask = np.triu(np.ones_like(corr, dtype=bool))   # upper triangle mask
 
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(
        corr, mask=mask, annot=True, fmt=".2f",
        cmap="RdYlGn", center=0, linewidths=0.4,
        annot_kws={"size": 8}, ax=ax,
        vmin=-1, vmax=1,
    )
    ax.set_title("Correlation Heatmap — All Numeric Columns", fontweight="bold", fontsize=13)
    plt.xticks(rotation=45, ha="right", fontsize=9)
    plt.yticks(rotation=0, fontsize=9)
    plt.tight_layout()
    plt.savefig("correlation_heatmap.png", bbox_inches="tight")
    plt.show()
    print("📁  Saved: correlation_heatmap.png")
plot_correlation_heatmap(data) 

In [ ]:
def plot_cases_vs_deaths(data: pd.DataFrame) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
 
    for ax, (x_col, y_col, title) in zip(axes, [
        ("Road_Cases",  "Road_Died",  "Road Accidents"),
        ("Rail_Cases",  "Rail_Died",  "Railway Accidents"),
    ]):
        ax.scatter(data[x_col], data[y_col], alpha=0.6, edgecolors="white", s=60)
 
        # Regression line
        mask = data[x_col].notna() & data[y_col].notna()
        m, b = np.polyfit(data.loc[mask, x_col], data.loc[mask, y_col], 1)
        x_line = np.linspace(data[x_col].min(), data[x_col].max(), 200)
        ax.plot(x_line, m * x_line + b, linewidth=1.5, linestyle="--")
 
        # Annotate top 3 states
        top3 = data.nlargest(3, y_col)
        for _, row in top3.iterrows():
            ax.annotate(row["State_UT"], xy=(row[x_col], row[y_col]),
                        xytext=(6, 4), textcoords="offset points", fontsize=8)
 
        corr = data[[x_col, y_col]].corr().iloc[0, 1]
        ax.set_title(f"{title} — Cases vs Deaths (r={corr:.2f})", fontweight="bold")
        ax.set_xlabel(x_col.replace("_", " "))
        ax.set_ylabel(y_col.replace("_", " "))
 
    plt.tight_layout()
    plt.savefig("scatter_cases_vs_deaths.png", bbox_inches="tight")
    plt.show()
    print("📁  Saved: scatter_cases_vs_deaths.png")
 
plot_cases_vs_deaths(data)

# State accident analysis

In [ ]:
# Top 10 States by Total Deaths 
top10 = data.nlargest(10, "Total_Died")[["State_UT", "Total_Died"]]
 
plt.figure(figsize=(9, 5))
sns.barplot(data=top10, y="State_UT", x="Total_Died", palette="Reds_r")
plt.title("Top 10 States — Total Deaths", fontweight="bold")
plt.xlabel("Deaths"); plt.ylabel("") 
plt.tight_layout(); plt.savefig("top10_deaths.png"); plt.show() 

In [ ]:
# Road vs Railway vs Crossing — Deaths Share (Pie) ──────────────────────
shares = {
    "Road"     : data["Road_Died"].sum(),
    "Railway"  : data["Rail_Died"].sum(),
    "Crossing" : data["Crossing_Died"].sum(),
}
 
plt.figure(figsize=(3, 3))
plt.pie(shares.values(), labels=shares.keys(), autopct="%1.1f%%", startangle=160)
plt.title("Deaths by Accident Type", fontweight="bold")
plt.tight_layout(); plt.savefig("death_share_pie.png"); plt.show()

# Phase 2 — Feature Engineering

In [ ]:
# Fatality Rates 
data["Road_Fatality_Rate"]     = data["Road_Died"]     / data["Road_Cases"].replace(0, np.nan) * 100
data["Rail_Fatality_Rate"]     = data["Rail_Died"]     / data["Rail_Cases"].replace(0, np.nan) * 100
data["Crossing_Fatality_Rate"] = data["Crossing_Died"] / data["Crossing_Cases"].replace(0, np.nan) * 100
data["Overall_Fatality_Rate"]  = data["Total_Died"]    / data["Total_Cases"].replace(0, np.nan) * 100

In [ ]:
# Injury Severity Rate 
data["Road_Injury_Rate"]    = data["Road_Injured"]  / data["Road_Cases"].replace(0, np.nan) * 100
data["Rail_Injury_Rate"]    = data["Rail_Injured"]  / data["Rail_Cases"].replace(0, np.nan) * 100

# ── 5. Road vs Rail Dominance flag ───────────────────────────────────────────
data["Dominant_Type"] = np.where(data["Road_Cases"] >= data["Rail_Cases"], "Road", "Railway")

In [ ]:
#  Accident Type Share (% of total cases) 
data["Road_Cases_Share"]     = data["Road_Cases"]     / data["Total_Cases"].replace(0, np.nan) * 100
data["Rail_Cases_Share"]     = data["Rail_Cases"]     / data["Total_Cases"].replace(0, np.nan) * 100
data["Crossing_Cases_Share"] = data["Crossing_Cases"] / data["Total_Cases"].replace(0, np.nan) * 100

# Crossing Death Share (hidden danger signal) 
data["Crossing_Death_Share"] = data["Crossing_Died"] / data["Total_Died"].replace(0, np.nan) * 100 

In [ ]:
# Composite Risk Score (weighted) 
# Weights: fatality rate 50%, injury rate 30%, crossing danger 20%
data["Composite_Risk_Score"] = (
    0.50 * data["Overall_Fatality_Rate"].fillna(0) +
    0.30 * data["Road_Injury_Rate"].fillna(0) +
    0.20 * data["Crossing_Death_Share"].fillna(0)
).round(2)  

# Risk Tier (High / Medium / Low) 
data["Risk_Tier"] = pd.qcut(
    data["Composite_Risk_Score"],
    q=3,
    labels=["Low", "Medium", "High"]
)

# Percentile Rank (0–100) 
data["Risk_Percentile"] = data["Composite_Risk_Score"].rank(pct=True).mul(100).round(1)

# Preview 
key_cols = [
    "State_UT", "Overall_Fatality_Rate", "Composite_Risk_Score",
    "Risk_Tier", "Risk_Percentile", "Dominant_Type"
] 
print(data[key_cols].sort_values("Composite_Risk_Score", ascending=False).to_string(index=False))

In [ ]:
high_risk = data[data["Risk_Tier"] == "High"]["State_UT"].tolist()
top_fr    = data.loc[data["Overall_Fatality_Rate"].idxmax(), "State_UT"]
top_fr_v  = data["Overall_Fatality_Rate"].max()
top_cross = data.loc[data["Crossing_Death_Share"].idxmax(), "State_UT"]
top_rs    = data.loc[data["Composite_Risk_Score"].idxmax(), "State_UT"]
 
print(f"\n• High-risk states       : {', '.join(high_risk[:5])}{'...' if len(high_risk)>5 else ''}")
print(f"• Highest fatality rate  : {top_fr} ({top_fr_v:.1f}% deaths per 100 cases)")
print(f"• Most crossing danger   : {top_cross} (highest crossing death share)")
print(f"• Top composite risk     : {top_rs}")
print(f"\n• Road dominates in      : {(data['Dominant_Type']=='Road').sum()} / {len(data)} states")
print(f"• Railway dominates in   : {(data['Dominant_Type']=='Railway').sum()} / {len(data)} states")
print(f"\n• Avg overall fatality   : {data['Overall_Fatality_Rate'].mean():.2f}%")
print(f"• Avg composite risk     : {data['Composite_Risk_Score'].mean():.2f}")
print("\n  → States in the HIGH tier need policy intervention most urgently.")
print("  → Crossing Death Share exposes danger invisible in raw counts.")
print("  → Composite Risk Score is your single best ranking metric for the dashboard.")
print("═"*55) 

In [ ]:
!pip install sqlalchemy psycopg2
print("psycopg2 package imported successfully!")
print("sqlalchemy installed succesfully") 

from sqlalchemy import create_engine

# Replace with your actual password
engine = create_engine("") 

data.to_sql("my_cleaned_data", con=engine, if_exists="replace", index=False)

print(pd.read_sql("SELECT * FROM my_cleaned_data;", con=engine)) 

df_summary = pd.read_sql("SELECT COUNT(*) FROM my_cleaned_data;", con=engine)
print(df_summary) 
